Based on:
https://github.com/ErwannMillon/Color-diffusion

# Libraries

In [1]:
!git clone https://github.com/ErwannMillon/Color-diffusion.git

fatal: destination path 'Color-diffusion' already exists and is not an empty directory.


In [2]:
%cd Color-diffusion

/content/Color-diffusion


In [3]:
!pip install -r requirements.txt

In [5]:
import pandas as pd

is_large_dataset = True

if is_large_dataset:
    server_port = 1986 # Large dataset of ~10K images
else:
    server_port = 1985 # Large dataset of ~10K images
# server_port = 1985 # initial dataset of 3.6K images

raw_data_csv_file_link = f"https://perritos.myasustor.com:{server_port}/metadata.csv"


metadata_raw_df = pd.read_csv(raw_data_csv_file_link, index_col=0)
metadata_raw_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10008 entries, 0 to 10007
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   img_id      10008 non-null  int64  
 1   img_name    10008 non-null  object 
 2   latitude    10008 non-null  float64
 3   longitude   10008 non-null  float64
 4   zoom_level  10008 non-null  int64  
 5   class       10008 non-null  int64  
 6   link        10008 non-null  object 
dtypes: float64(2), int64(3), object(2)
memory usage: 625.5+ KB


In [24]:
import requests
from io import BytesIO
from PIL import Image
from skimage.color import rgb2lab
from torch.utils.data import Dataset, DataLoader
import torch
import numpy as np
from torchvision import transforms

class ColorizationDataset(Dataset):
    def __init__(self, img_ids, server_port, split='train', config=None):
        size = config["img_size"]
        self.server_port = server_port
        self.resize = transforms.Resize((size, size), Image.BICUBIC)
        if split == 'train':
            self.transforms = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(brightness=0.3,
                                       contrast=0.1,
                                       saturation=(1., 2.),
                                       hue=0.05),
                self.resize
            ])
        elif split == 'val':
            self.transforms = self.resize
        self.img_ids = img_ids

    def tensor_to_lab(self, base_img_tens):
        base_img = np.array(base_img_tens)
        img_lab = rgb2lab(base_img).astype(
            "float32")  # Converting RGB to L*a*b
        img_lab = transforms.ToTensor()(img_lab)
        L = img_lab[[0], ...] / 50. - 1.  # Between -1 and 1
        ab = img_lab[[1, 2], ...] / 110.  # Between -1 and 1
        return torch.cat((L, ab), dim=0)

    def _fetch_image(self, img_id):
        img_in_server_link = f"https://perritos.myasustor.com:{self.server_port}/data/img_id_{img_id}.jpg"
        response = requests.get(img_in_server_link)
        image = Image.open(BytesIO(response.content)).convert("RGB")
        return image

    def __getitem__(self, index):
        img_id = self.img_ids[index]
        image = self._fetch_image(img_id)
        image = self.transforms(image)
        image = self.tensor_to_lab(image)
        return image

    def __len__(self):
        return len(self.img_ids)

In [25]:
# Step 3: Create Data Loaders
def make_dataloaders(server_port, img_ids, config, num_workers=2, limit=None):
    if limit is not None:
        img_ids = img_ids[:limit]

    split_index = int(len(img_ids) * 0.8)  # 80-20 train-validation split
    train_img_ids = img_ids[:split_index]
    val_img_ids = img_ids[split_index:]

    train_dataset = ColorizationDataset(train_img_ids, server_port, split='train', config=config)
    val_dataset = ColorizationDataset(val_img_ids, server_port, split='val', config=config)

    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=num_workers)

    return train_loader, val_loader

In [30]:
!pip install ema-pytorch diffusion-pytorch

ERROR: Could not find a version that satisfies the requirement diffusion-pytorch (from versions: none)
ERROR: No matching distribution found for diffusion-pytorch


In [29]:
# Step 4: Define the Model
import pytorch_lightning as pl
import torch
import torch.nn.functional as F
from ema_pytorch import ExponentialMovingAverage
from diffusion_pytorch import GaussianDiffusion
import torchvision
import matplotlib.pyplot as plt

class ColorDiffusion(pl.LightningModule):
    def __init__(self,
                 unet,
                 train_dl,
                 val_dl,
                 encoder,
                 loss_fn="l2",
                 T=300,
                 lr=1e-4,
                 batch_size=12,
                 sample=True,
                 should_log=True,
                 using_cond=False,
                 display_every=None,
                 dynamic_threshold=False,
                 use_ema=True,
                 **kwargs):
        super().__init__()
        self.unet = unet.to(self.device)
        self.T = T
        self.lr = lr
        self.using_cond = using_cond
        self.sample = sample
        self.should_log = should_log
        self.encoder = encoder
        self.display_every = display_every
        self.val_dl = val_dl
        self.train_dl = train_dl
        if loss_fn == "l1":
            self.loss_fn = torch.nn.functional.l1_loss
        else:
            self.loss_fn = torch.nn.functional.mse_loss

        self.ema = ExponentialMovingAverage(self.unet.parameters(),
                                            decay=0.9999)
        self.ema.to(self.device)
        self.diffusion = GaussianDiffusion(T,
                                           dynamic_threshold=dynamic_threshold)
        if sample is True and display_every is None:
            display_every = 1000
        self.save_hyperparameters(ignore=['unet'])

    def forward(self, x_noised, t, x_l):
        """
        Performs one denoising step on batch of noised inputs
        Unet is conditioned on timestep and features extracted from greyscale channel
        """
        cond = self.encoder(x_l)
        noise_pred = self.unet(x_noised, t, greyscale_embs=cond)
        return noise_pred

    def get_batch_pred(self, x_0, x_l):
        """
        Samples a timestep from range [0, T]
        Adds noise to images x_0 to get x_t (x_0 with color channels noised)
        Returns:
        - The model's prediction of the noise,
        - The real noise applied to the color channels by the forward diffusion process
        """
        t = torch.randint(0, self.T, (x_0.shape[0],)).to(x_0)
        x_noised, noise = self.diffusion.forward_diff(x_0, t, T=self.T)
        return (self(x_noised, t, x_l), noise)

    def get_losses(self, noise_pred, noise, x_l):
        diff_loss = self.loss_fn(noise_pred, noise)
        return {"total loss": diff_loss}

    def training_step(self, x_0, batch_idx):
        x_l, _ = split_lab_channels(x_0)
        noise_pred, noise = self.get_batch_pred(x_0, x_l)
        losses = self.get_losses(noise_pred, noise, x_l)
        self.log_dict(losses, on_step=True)
        if self.sample and batch_idx and batch_idx % self.display_every == 0 and self.global_step > 1:
            self.test_step(x_0)
        return losses["total loss"]

    def validation_step(self, batch, batch_idx):
        x_l, _ = split_lab_channels(batch)
        noise_pred, noise = self.get_batch_pred(batch, x_l)
        losses = self.get_losses(noise_pred, noise, x_l)
        if self.should_log:
            self.log("val_loss", losses["total loss"])
        if self.sample and batch_idx and batch_idx % self.display_every == 0:
            self.sample_plot_image(batch)
        return losses["total loss"]

    @torch.inference_mode()
    def test_step(self, batch, *args, **kwargs):
        x = next(iter(self.val_dl)).to(batch)
        self.sample_plot_image(x)
        self.sample_plot_image(x, use_ema=True)

    def configure_optimizers(self):
        learnable_params = list(self.unet.parameters()) \
                            + list(self.encoder.parameters())
        global_optim = torch.optim.AdamW(learnable_params,
                                         lr=self.lr,
                                         weight_decay=28e-3)
        return global_optim

    def log_img(self, image, caption="diff samples", use_ema=False):
        rgb_imgs = lab_to_rgb(*split_lab_channels(image))
        if use_ema:
            self.logger.log_image("EMA samples", [rgb_imgs])
        else:
            self.logger.log_image("samples", [rgb_imgs])

    def on_before_zero_grad(self, *args, **kwargs):
        self.ema.update()

    @torch.inference_mode()
    def sample_loop(self, x_l, prog=False, use_ema=False, save_all=False):
        """
        Noises color channels to timestep T, then denoises the color channels
        to t=0 to get the colorized image.
        Returns an array containing the noised image,
        intermediate images in the denoising process, and the final image
        """
        ema = self.ema if use_ema else None
        images = []
        num_images = 13
        img_size = x_l.shape[-1]
        stepsize = self.T // num_images

        # Initialize image with random noise in color channels
        x_ab = torch.randn((x_l.shape[0], 2, img_size, img_size)).to(x_l)
        img = torch.cat((x_l, x_ab), dim=1)

        counter = range(0, self.T)[::-1]
        if prog:
            counter = tqdm(counter)
        for i in counter:
            t = torch.full((1,), i, dtype=torch.long).to(img)

            img = self.diffusion.sample_timestep(self.unet,
                                                 self.encoder,
                                                 img,
                                                 t,
                                                 T=self.T,
                                                 cond=x_l,
                                                 ema=ema)
            if i % stepsize == 0:
                images += img.unsqueeze(0)
            if save_all and i % 2 == 0:
                pil_img = lab_to_pil(img)
                pil_img.save(f"./visualization/denoising/{i:04d}.png")
        return images

    @torch.inference_mode()
    def sample_plot_image(self, x_0, show=True, prog=False,
                          use_ema=False, log=True, save_all=False):
        """
        Denoises a single image and displays a grid showing:
        - ground truth image
        - intermediate denoised outputs
        - the final denoised image
        """
        print("Sampling image")
        ground_truth_images = []
        if x_0.shape[1] == 3:
            x_l, _ = split_lab_channels(x_0)
            ground_truth_images.append(x_0[:1])
        else:
            x_l = x_0
        x_l = x_l[:1]
        greyscale = torch.cat((x_l, *[torch.zeros_like(x_l)] * 2), dim=1)
        ground_truth_images += greyscale.unsqueeze(0)
        if len(x_l.shape) == 3:
            x_l = x_l.unsqueeze(0)
        images = ground_truth_images + self.sample_loop(x_l,
                                                        prog=prog,
                                                        use_ema=use_ema,
                                                        save_all=save_all)
        grid = torchvision.utils.make_grid(torch.cat(images), dim=0).to(x_l)
        if show:
            show_lab_image(grid.unsqueeze(0), log=self.should_log)
            plt.show()
        if self.should_log and log:
            self.log_img(grid.unsqueeze(0), use_ema=use_ema)
        return lab_to_rgb(*split_lab_channels(images[-1]))

ImportError: cannot import name 'ExponentialMovingAverage' from 'ema_pytorch' (/usr/local/lib/python3.10/dist-packages/ema_pytorch/__init__.py)

In [28]:
import wandb
from utils import load_default_configs
from pytorch_lightning.loggers import WandbLogger
from denoising import Unet, Encoder

# Define arguments directly
log = False
cpu_only = False
dataset_path = "./img_align_celeba"
ckpt = None
server_port = 1986

# Load configurations
enc_config, unet_config, colordiff_config = load_default_configs()

# Ensure 'n_steps' is in the configuration
if 'n_steps' not in colordiff_config:
    colordiff_config['n_steps'] = 10  # Set to a default value, adjust as needed

print(colordiff_config)  # Verify the configuration

# Load metadata
metadata_raw_df = pd.read_csv(f"https://perritos.myasustor.com:{server_port}/metadata.csv", index_col=0)
img_ids = metadata_raw_df.index.tolist()

# Create data loaders
train_dl, val_dl = make_dataloaders(server_port, img_ids, colordiff_config, num_workers=2, limit=35000)
colordiff_config["sample"] = False
colordiff_config["should_log"] = log

# Initialize model components
encoder = Encoder(**enc_config)
unet = Unet(**unet_config)

# Load or initialize model
if ckpt is not None:
    print(f"Resuming training from checkpoint: {ckpt}")
    model = ColorDiffusion.load_from_checkpoint(
        ckpt,
        strict=True,
        unet=unet,
        encoder=encoder,
        train_dl=train_dl,
        val_dl=val_dl,
        **colordiff_config
    )
else:
    model = ColorDiffusion(unet=unet,
                           encoder=encoder,
                           train_dl=train_dl,
                           val_dl=val_dl,
                           **colordiff_config)

# Set up trainer
if log:
    wandb_logger = WandbLogger(project="Color_diffusion_v2")
    wandb_logger.watch(unet)
    wandb_logger.experiment.config.update(enc_config)
    wandb_logger.experiment.config.update(unet_config)
    wandb_logger.experiment.config.update(colordiff_config)
    trainer = pl.Trainer(logger=wandb_logger, max_epochs=colordiff_config["n_steps"])
else:
    trainer = pl.Trainer(max_epochs=colordiff_config["n_steps"])

{'device': 'auto', 'pin_memory': True, 'T': 350, 'lr': 1e-06, 'loss_fn': 'l2', 'batch_size': 36, 'accumulate_grad_batches': 2, 'img_size': 64, 'sample': True, 'should_log': True, 'epochs': 14, 'using_cond': True, 'display_every': 350, 'dynamic_threshold': False, 'train_autoenc': False, 'enc_loss_coeff': 1.1, 'n_steps': 10}


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
